Classify whether the text is about a disaster or not

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

Make sure to upload [csv file](https://www.kaggle.com/datasets/vstepanenko/disaster-tweets) for this to work:


In [4]:
# create dataframe from file
df = pd.read_csv("tweets.csv")

df.head()

,id,keyword,location,text,target
0,0,ablaze,NaN,"Communal violence in Bhainsa, Telangana. ""Ston...",1
1,1,ablaze,NaN,Telangana: Section 144 has been imposed in Bha...,1
2,2,ablaze,New York City,Arsonist sets cars ablaze at dealership https:...,1
3,3,ablaze,"Morgantown, WV",Arsonist sets cars ablaze at dealership https:...,1
4,4,ablaze,NaN,"""Lord Jesus, your love brings freedom and pard...",0


According to where I found the [dataset](https://www.kaggle.com/datasets/vstepanenko/disaster-tweets), no rows have an empty text or target column

In [10]:
small_df = df[["text","target"]]
small_df.head()

,text,target
0,"Communal violence in Bhainsa, Telangana. ""Ston...",1
1,Telangana: Section 144 has been imposed in Bha...,1
2,Arsonist sets cars ablaze at dealership https:...,1
3,Arsonist sets cars ablaze at dealership https:...,1
4,"""Lord Jesus, your love brings freedom and pard...",0


In [11]:
train, val, test = np.split(small_df.sample(frac=1), [int(0.8*len(small_df)), int(0.9*len(small_df))])



/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


helper function to change the pandas df to a dataset TensorFlow can use

In [16]:
def df_to_dataset(dataframe, shuffle=True, batch_size=1024):
  df = dataframe.copy()
  labels = df.pop('target')
  df = df["text"]
  ds = tf.data.Dataset.from_tensor_slices((df, labels))
  if shuffle:
    ds = ds.shuffle(buffer_size=len(dataframe))
  ds = ds.batch(batch_size)
  ds = ds.prefetch(tf.data.AUTOTUNE)
  return ds

In [17]:
train_data = df_to_dataset(train)
valid_data = df_to_dataset(val)
test_data = df_to_dataset(test)

In [19]:
# create encoder: learns strings of text from the data and turns them to a vocab
# (numbers) that a model can understand
encoder = tf.keras.layers.TextVectorization(max_tokens=2000)
encoder.adapt(train_data.map(lambda text, label: text))
vocab = np.array(encoder.get_vocabulary())


define model types (seqyential) amd layers

In [21]:
model = tf.keras.Sequential([
    # these layers take the text and encodes them
    encoder,
    tf.keras.layers.Embedding(
        input_dim=len(encoder.get_vocabulary()),
        output_dim=32,
        mask_zero=True
    ),

    # Long Short Term Memory
    tf.keras.layers.LSTM(32),

    # Standard layers
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.4),  # prevents overfitting by dropping random neurons
    tf.keras.layers.Dense(1, activation='sigmoid')
])

In [25]:
# compile model with specified optimizer, loss function, it will be judged by accuracy
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss=tf.keras.losses.BinaryCrossentropy(),
              metrics=['accuracy'])

In [26]:
# test model initially, should not be too good since it is random
model.evaluate(train_data)
model.evaluate(valid_data)

9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.8071 - loss: 0.6899
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8108 - loss: 0.6897


[0.6896964907646179, 0.8109058737754822]

honestly that was better than I expected. Anyways, lets see how good it can get after training!

In [30]:
# TRAIN MODEL, keep track of history at each stage in case it is important later
history = model.fit(train_data, epochs=10, validation_data=valid_data)

Epoch 1/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 114ms/step - accuracy: 0.8167 - loss: 0.4674 - val_accuracy: 0.8206 - val_loss: 0.4380
Epoch 2/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 136ms/step - accuracy: 0.8123 - loss: 0.4519 - val_accuracy: 0.8206 - val_loss: 0.4097
Epoch 3/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 2s 206ms/step - accuracy: 0.8181 - loss: 0.4078 - val_accuracy: 0.8285 - val_loss: 0.3514
Epoch 4/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 2s 114ms/step - accuracy: 0.8264 - loss: 0.3642 - val_accuracy: 0.8725 - val_loss: 0.3143
Epoch 5/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 116ms/step - accuracy: 0.8755 - loss: 0.3270 - val_accuracy: 0.8830 - val_loss: 0.2987
Epoch 6/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 114ms/step - accuracy: 0.8874 - loss: 0.3003 - val_accuracy: 0.8795 - val_loss: 0.2875
Epoch 7/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 114ms/step - accuracy: 0.8988 - loss: 0.2811 - val_accuracy: 0.8918 - val_loss: 0.2768
Epoch 8/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 114ms/step - accuracy: 0.9117 - loss: 0.2535 - val_accuracy: 0.8909 - val_loss:

In [31]:
# FINAL TEST, how good is out disaster (or not) classifier
model.evaluate(test_data)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8714 - loss: 0.3551


[0.35228413343429565, 0.8724713921546936]

In [48]:
test_tweet = tf.constant(["Everybody be careful, there is a magnitude 3 earthquake in Dallas."])


In [49]:
prediction = model.predict(test_tweet)

print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
[[0.53299713]]
